<a href="https://colab.research.google.com/github/Atif-04/atifkhaleel_augustbatch/blob/main/q3_shopping_cart_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Q3 - Mutable Default Arguments & Pass-by-Reference
====================================================

PART A — Spot the Bug
----------------------
def add_item(item, cart=[]):
    cart.append(item)
    return cart

Predicted output:
    print(add_item("apple"))                   -> ['apple']
    print(add_item("banana"))                   -> ['apple', 'banana']
    print(add_item("milk", cart=["bread"]))      -> ['bread', 'milk']
    print(add_item("eggs"))                      -> ['apple', 'banana', 'eggs']

WHY this happens:
    Default argument values in Python are evaluated ONCE, at function
    DEFINITION time (not each time the function is called), and the
    resulting object is stored as part of the function itself.

    So `cart=[]` creates ONE list object when `add_item` is defined.
    Every call that does NOT explicitly pass `cart` reuses that exact
    same list object. Since lists are mutable, `.append()` modifies it
    in place, and the changes persist across calls, silently
    accumulating data between unrelated calls.

    The 3rd call passes its own list (["bread"]) explicitly, so it
    does not touch the shared default list at all. The 4th call
    (no cart passed) goes right back to using the same shared default
    list from calls 1 and 2, which already contains ["apple", "banana"].
"""


def add_item_buggy(item, cart=[]):
    """
    The original buggy version from Part A, kept under its own name
    so we can actually demonstrate the bug live in main() without it
    being shadowed by the fixed version below.
    """
    cart.append(item)
    return cart


# ---------------------------------------------------------------
# PART B — The Fix
# ---------------------------------------------------------------
def add_item(item, cart=None):
    """
    Correct version: use None as the default sentinel, and create a
    brand-new list inside the function body every time none was
    supplied. This guarantees a fresh, independent list per call
    (unless the caller deliberately passes one in to reuse).
    """
    if cart is None:
        cart = []
    cart.append(item)
    return cart


# ---------------------------------------------------------------
# PART C — Shopping Cart Program
# ---------------------------------------------------------------
def create_cart(owner, discount=0):
    """
    Create a new cart for `owner`.
    discount=0 is safe as a default because integers (and other
    immutables) can never be mutated in place -- every cart gets
    its own fresh 'items' list created fresh inside this call.
    """
    return {"owner": owner, "items": [], "discount": discount}


def add_to_cart(cart, name, price, qty=1):
    """Append a new item dict to this specific cart's item list."""
    cart["items"].append({"name": name, "price": price, "qty": qty})
    return cart


def update_price(price_tuple, new_price):
    """
    Attempt to modify an element of a tuple.

    Tuples are IMMUTABLE: once created, their internal references to
    elements cannot be reassigned. There is no in-place '__setitem__'
    support for tuples, so trying `price_tuple[0] = new_price` raises
    a TypeError. This immutability is intentional -- it's what makes
    tuples hashable (usable as dict keys / set members) and safe to
    share around a program without fear of accidental modification.
    """
    try:
        price_tuple[0] = new_price  # illegal - tuples don't support item assignment
        return price_tuple
    except TypeError as e:
        print(f"TypeError caught as expected: {e}")
        return price_tuple  # unchanged, original tuple returned


def calculate_total(cart):
    """Sum price*qty over all items, then apply the cart's discount %."""
    subtotal = sum(item["price"] * item["qty"] for item in cart["items"])
    discount_amount = subtotal * (cart["discount"] / 100)
    total = subtotal - discount_amount
    return round(total, 2)


def main():
    # --- Demonstrate Part A bug live ---
    print("=== Part A: Buggy behavior demo ===")
    print(add_item_buggy("apple"))
    print(add_item_buggy("banana"))
    print(add_item_buggy("milk", cart=["bread"]))
    print(add_item_buggy("eggs"))

    # --- Demonstrate Part B fix ---
    print("\n=== Part B: Fixed add_item (fresh list each time) ===")
    print(add_item("apple"))   # fresh list -> ['apple']
    print(add_item("banana"))  # fresh list -> ['banana'], NOT ['apple', 'banana']

    # --- Part C: Two independent customer carts ---
    print("\n=== Part C: Independent Shopping Carts ===")
    cart_alice = create_cart("Alice", discount=10)
    cart_bob = create_cart("Bob")  # discount=0 default, safe

    add_to_cart(cart_alice, "Bread", 2.50, qty=2)
    add_to_cart(cart_alice, "Milk", 1.80)

    add_to_cart(cart_bob, "Eggs", 3.00, qty=2)
    add_to_cart(cart_bob, "Cheese", 4.50)

    print(f"Alice's cart: {cart_alice}")
    print(f"Bob's cart:   {cart_bob}")

    # Proof the carts don't share the same 'items' list object:
    print(f"\nSame items list object? {cart_alice['items'] is cart_bob['items']}")

    # Mutate Alice's cart directly -- Bob's must be unaffected
    cart_alice["items"].append({"name": "Butter", "price": 2.20, "qty": 1})
    print(f"\nAfter adding Butter to Alice's cart only:")
    print(f"Alice's items: {[i['name'] for i in cart_alice['items']]}")
    print(f"Bob's items:   {[i['name'] for i in cart_bob['items']]}")

    # Totals
    print(f"\nAlice's total (10% discount): {calculate_total(cart_alice)}")
    print(f"Bob's total (no discount):    {calculate_total(cart_bob)}")

    # --- Tuple immutability demo ---
    print("\n=== Tuple Immutability Demo ===")
    price_info = ("Bread", 2.50)
    update_price(price_info, 3.00)
    print(f"Tuple after failed update attempt (unchanged): {price_info}")


if __name__ == "__main__":
    main()


# =================================================================
# DISCUSSION POINTS
# =================================================================
#
# 1. Why is discount=0 safe but cart=[] dangerous?
#    ------------------------------------------------
#    `discount=0` is an int, which is IMMUTABLE. There is no operation
#    that can change an int "in place" -- any operation that looks like
#    it modifies it (e.g. discount += 5) actually creates a brand-new
#    int object and rebinds the name to it. The original default object
#    stored on the function is never touched, so every call still sees
#    the same untouched value 0.
#
#    `cart=[]` is a list, which is MUTABLE. The default list object is
#    created once at function-definition time and reused across every
#    call that doesn't override it. Methods like .append() mutate that
#    SAME object in place, so changes leak between unrelated calls --
#    exactly the bug demonstrated in Part A.
#
# 2. What is the difference between rebinding and mutating?
#    ------------------------------------------------
#    Rebinding: pointing a variable name at a NEW object.
#        x = [1, 2]
#        x = [3, 4]      # rebinding -- x now points to a different list
#    The original [1, 2] object is untouched (and gets garbage collected
#    if nothing else references it).
#
#    Mutating: changing the CONTENTS of an existing object in place,
#    without creating a new object.
#        x = [1, 2]
#        x.append(3)      # mutating -- same list object, new contents
#    Any other variable that also referenced that same list object will
#    see the change too, because they're both pointing at the one object.
#
# 3. Which of these are mutable? -- list, tuple, dict, set, str, int
#    ------------------------------------------------
#    Mutable:    list, dict, set
#    Immutable:  tuple, str, int
#
# 4. When you pass a list into a function and modify it, do changes
#    reflect outside? Why?
#    ------------------------------------------------
#    Yes. Python passes arguments by "assignment" (often called
#    "pass-by-object-reference"): the parameter inside the function
#    becomes another name bound to the SAME object the caller passed in.
#    If the function MUTATES that object (e.g. list.append, dict[key]=x,
#    set.add), the caller sees the change too, since there's only one
#    underlying object being shared.
#    However, if the function REBINDS the parameter to a new object
#    (e.g. cart = [] inside the function), that only changes what the
#    local name points to -- the caller's original object is untouched.

=== Part A: Buggy behavior demo ===
['apple']
['apple', 'banana']
['bread', 'milk']
['apple', 'banana', 'eggs']

=== Part B: Fixed add_item (fresh list each time) ===
['apple']
['banana']

=== Part C: Independent Shopping Carts ===
Alice's cart: {'owner': 'Alice', 'items': [{'name': 'Bread', 'price': 2.5, 'qty': 2}, {'name': 'Milk', 'price': 1.8, 'qty': 1}], 'discount': 10}
Bob's cart:   {'owner': 'Bob', 'items': [{'name': 'Eggs', 'price': 3.0, 'qty': 2}, {'name': 'Cheese', 'price': 4.5, 'qty': 1}], 'discount': 0}

Same items list object? False

After adding Butter to Alice's cart only:
Alice's items: ['Bread', 'Milk', 'Butter']
Bob's items:   ['Eggs', 'Cheese']

Alice's total (10% discount): 8.1
Bob's total (no discount):    10.5

=== Tuple Immutability Demo ===
TypeError caught as expected: 'tuple' object does not support item assignment
Tuple after failed update attempt (unchanged): ('Bread', 2.5)
